In [2]:
### Import library
import os
import numpy as np
import matplotlib.pyplot as plt
import copy
from mpl_toolkits.mplot3d import Axes3D
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torchvision.utils import save_image
import PIL
print(PIL.__version__)
from PIL import Image
print(Image.__version__)
from matplotlib.ticker import MultipleLocator

11.1.0
11.1.0


In [10]:
### Checking state
print("CUDA Available:", torch.cuda.is_available())  # True == available
print("cuDNN Available:", torch.backends.cudnn.is_available())  # True == available
print("cuDNN Version:", torch.backends.cudnn.version())  # print cuDNN version

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

CUDA Available: True
cuDNN Available: True
cuDNN Version: 90100
Using device: cuda


In [ ]:
### Data load
class VoxelDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.files = [f for f in os.listdir(data_dir) if f.endswith(".npy")]
        # count num of .npy files in data_dir

        self.labels = [int(f.split("_")[1]) for f in self.files] 

    def __len__(self):
        return len(self.files) # num of total data sample
    
    def __getitem__(self, idx): # return idx(th) sample
        file_path = os.path.join(self.data_dir, self.files[idx])
        voxel_data = np.load(file_path) # load file (.npy)
        voxel_data = torch.tensor(voxel_data, dtype = torch.float32).unsqueeze(0) # add channel
        label = torch.tensor(self.labels[idx], dtype = torch.long) # change label to tensor
        
        return voxel_data, label
    

dataset = VoxelDataset("samples/")
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

# Print a first batch
for batch, labels in dataloader:
    batch = batch.to(device)
    labels = labels.to(device)
    print(batch.device, labels.device)
    print(f"Voxel shape: {batch.shape}, Label: {labels.shape}")
    # if you need visualization, change device position
    break

cuda:0 cuda:0
Voxel shape: torch.Size([32, 1, 256, 64, 64]), Label: torch.Size([32])
torch.Size([32])
torch.Size([32, 1])


In [ ]:
### Define labeling function 

class Labeling():
    def __init__(self, num_labels = 4):
        self.num_labels = num_labels

    def onehot(self, labels):
        rlabel = labels.unsqueeze(1) # [1,2,3] -> [[1],[2],[3]]
        tlabel = torch.zeros(rlabel.size(0), self.num_labels).to(device)

        for i in range(rlabel.size(0)):
            if rlabel[i] == 0:
                tlabel[i][0] = 1
            elif rlabel[i] == 15:
                tlabel[i][1] = 1
            elif rlabel[i] == 30:
                tlabel[i][2] = 1
            else:
                tlabel[i][3] = 1

        return tlabel
    
    def angle(self, labels):
        radian = torch.deg2rad(labels)
        cos = torch.cos(radian)
        sin = torch.sin(radian)

        return torch.stack([cos,sin], dim = 1).to(device)
    
    def multiple(self, lables):
        onehot_labels = self.onehot()
        algle_labels = self.angle()
        add_labels = torch.cat((onehot_labels, algle_labels), dim = 1)

        return add_labels

In [37]:
### Define visualization function

class Visualization():
    def __init__(self, cpoint = 'red', cbackground = 'none', mshape = '.', xzview = [30,45]):
        self.point = cpoint
        self.background = cbackground
        self.mshape = mshape
        self.xzview = xzview

    def save(self, voxel, title):
        fig = plt.figure()
        ax = fig.add_subplot(111, projection = '3d')

        x_all, y_all, z_all = [64,64,256]
        colors = np.where(voxel.flatten() != 0, self.point, self.background)
        ax.scatter(x_all.flatten(), y_all.flatten(), z_all.flatten(), c = colors, marker = '.')
        ax.view_init(elev = self.xzview[0], azim = self.xzview[1])

        limits = [ax.get_xlim(), ax.get_ylim(), ax.get_zlim()] # axis fix
        ax.set_box_aspect([np.ptp(limit) for limit in limits])

        plt.title(f"{title}")
        plt.savefig(r"D:\research\7. TSP_3\20250216 python용 onepool 추출\augmentation\training\{}.png".format(title))
        plt.close()

In [ ]:
# Define accuracy function
def compute_acc(preds, labels):
    correct = 0
    preds_ = preds.data.max(1)[1]
    correct = preds_.eq(labels.data).cpu().sum()
    acc = float(correct) / float(len(labels.data)) * 100.0
    return acc

In [3]:
### Generator

class Generator(torch.nn.Module):
    def __init__(self, in_channels = 256, in_shape = [16,4,4], out_channels = 1, latent_dim = 400, num_labels = 4):
        super(Generator, self).__init__()
        self.in_channels = in_channels
        self.in_shape = in_shape
        self.out_channels = out_channels
        self.latent_dim = latent_dim
        self.num_labels = num_labels

        channels_128ch = int(in_channels / 2) # 128
        channels_64ch = int(in_channels / 4) # 64
        channels_32ch = int(in_channels/ 8) # 32
        channels_16ch = int(in_channels / 16) # 16
        channels_8ch = int(in_channels / 32) # 8
        channels_4ch = int(in_channels / 64) # 4

        self.linear = torch.nn.Linear(latent_dim + num_labels, in_channels*in_shape[0] *in_shape[1]*in_shape[2])
        
        
        self.conv1 = nn.Sequential(         # (16, 4, 4) -> (32, 8, 8)
            nn.ConvTranspose3d(in_channels, channels_128ch, kernel_size = 4,stride = 2, padding = 1, bias = False),
            nn.BatchNorm3d(channels_128ch),
            nn.ReLU(inplace = True)
        )
        self.conv2 = nn.Sequential(         #(32,8,8) -> (64,16,16)
            nn.ConvTranspose3d(channels_128ch, channels_32ch, kernel_size = 4, stride = 2, padding = 1, bias = False),
            nn.BatchNorm3d(channels_32ch),
            nn.ReLU(inplace = True)
        )
        self.conv3 = nn.Sequential(         #(64,16,16) -> (128,32,32)
            nn.ConvTranspose3d(channels_32ch, channels_8ch, kernel_size = 4, stride = 2, padding = 1, bias = False),
            nn.BatchNorm3d(channels_8ch),
            nn.ReLU(inplace = True)
        )
        #self.conv4 = nn.Sequential(         #(128,32,32) -> (256,64,64)
        #    nn.ConvTranspose3d(channels_8ch, out_channels, kernel_size = 4, stride = 2, padding = 1, bias = False),
        #    nn.BatchNorm3d(conv6_out_channels),
        #    nn.ReLU(inplace = True)
        #)

        self.replication_pad = nn.ReplicationPad3d(1)

    def forward(self, noise, labels):
        
        # If you watch the training process, delete the annotation
        #viz.save(self, noise.cpu().numpy(), title = "1. Input Data_16x4x4")
        
        # Label concat to sample
        trans_labels = Labeling.angle(self, labels)
        print(f"trans labels state: {trans_labels.shape} {trans_labels.device}")
        x = torch.cat((noise, trans_labels), dim=1)
        print(f"After labels concat: {x.shape}")

        x = self.linear(x)
        x = x.view(-1, self.in_channels, self.in_shape[0], self.in_shape[1], self.in_shape[2])
        #viz.save(self, x.cpu().numpy(), title = "2. Labeling and Linear")

        x = self.conv1(x)
        #viz.save(self, x.cpu().numpy(), title = "3. 3D ConvT_32x8x8")
        x = x[:, :, 1 : -1, 1 : -1, 1 : -1]
        #viz.save(self, x.cpu().numpy(), title = "4. Peel off_32x8x8")
        x = self.replication_pad(x)
        #viz.save(self, x.cpu().numpy(), title = "5. Padding_32x8x8")
        print(f"After conv1: {x.shape}")
        
        x = self.conv2(x)
        #viz.save(self, x.cpu().numpy(), title = "6. 3D ConvT_64x16x16")
        x = x[:, :, 1 : -1, 1 : -1, 1 : -1]
        #viz.save(self, x.cpu().numpy(), title = "7. Peel off_64x16x16")
        x = self.replication_pad(x)
        #viz.save(self, x.cpu().numpy(), title = "8. Padding_64x16x16")
        print(f"After conv2: {x.shape}")
       
        x = self.conv3(x)
        #viz.save(self, x.cpu().numpy(), title = "9. 3D ConvT_128x32x32")
        x = x[:, :, 1 : -1, 1 : -1, 1 : -1]
        #viz.save(self, x.cpu().numpy(), title = "10. Peel off_128x32x32")
        x = self.replication_pad(x)
        #viz.save(self, x.cpu().numpy(), title = "11. Padding_128x32x32")
        print(f"After conv3: {x.shape}")
        
        upsample = torch.nn.Upsample(size=None, scale_factor=2, mode='nearest')
        # 'nearest', 'linear', 'bilinear', 'bicubic' and 'trilinear'
        x = upsample(x)
        
        return x

In [39]:
### check Generator shape
## Uncomment print() line in the Generator class

generator = Generator().to(device)
noise = torch.randn(batch.size(0), 400).to(device)

with torch.no_grad():
    test = generator(noise,labels)
    print(test.shape)


trans labels state: torch.Size([32, 4]) cuda:0
After labels concat: torch.Size([32, 404])
After conv1: torch.Size([32, 128, 32, 8, 8])
After conv2: torch.Size([32, 32, 64, 16, 16])
After conv3: torch.Size([32, 8, 128, 32, 32])
After conv4: torch.Size([32, 1, 256, 64, 64])
torch.Size([32, 1, 256, 64, 64])


In [4]:
### Discriminator

class Discriminator(torch.nn.Module):
    def __init__(self, in_channels = 1, out_channels = 256, out_shape = [16,4,4], num_labels = 4):
        super(Discriminator, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.out_shape = out_shape
        self.num_labels = num_labels

        channels_128ch = int(out_channels / 2) # 128
        channels_64ch = int(out_channels / 4) # 64
        channels_32ch = int(out_channels/ 8) # 32
        channels_16ch = int(out_channels / 16) # 16
        channels_8ch = int(out_channels / 32) # 8
        channels_4ch = int(out_channels / 64) # 4
        
        self.conv1 = nn.Sequential(         #(256,64,64) -> (128,32,32)
            nn.Conv3d(in_channels, channels_8ch, kernel_size = 4, stride = 2, padding = 1, bias = False),
            nn.BatchNorm3d(channels_8ch),
            nn.LeakyReLU(0.2, inplace = True)
        )
        self.conv2 = nn.Sequential(         #(128,32,32) -> (64,16,16)
            nn.Conv3d(channels_8ch, channels_32ch, kernel_size = 4, stride = 2, padding = 1, bias = False),
            nn.BatchNorm3d(channels_32ch),
            nn.LeakyReLU(0.2, inplace = True)
        )
        self.conv3 = nn.Sequential(         #(64,16,16) -> (32,8,8)
            nn.Conv3d(channels_32ch, channels_128ch, kernel_size = 4, stride = 2, padding = 1, bias = False),
            nn.BatchNorm3d(channels_128ch),
            nn.LeakyReLU(0.2, inplace = True)
        )
        self.conv4 = nn.Sequential(         #(32,8,8) -> (16,4,4)
            nn.Conv3d(channels_128ch, out_channels, kernel_size = 4, stride = 2, padding = 1, bias = False),
            nn.BatchNorm3d(out_channels),
            nn.LeakyReLU(0.2, inplace = True)
        )

        self.fc_dis = nn.Linear(out_channels*out_shape[0]*out_shape[1]*out_shape[2], 1)
        # aux-classifier fc
        self.fc_aux = nn.Linear(out_channels*out_shape[0]*out_shape[1]*out_shape[2], num_labels)
        # softmax and sigmoid
        self.softmax = nn.Softmax()
        self.sigmoid = nn.Sigmoid()

        #self.out = nn.Sequential(
        #    nn.Linear((out_channels*out_shape[0]*out_shape[1]*out_shape[2])+num_labels, 1),
        #    nn.Sigmoid(),
        #)

    def forward(self, voxel, labels):
        
        # If you watch the training process, delete the annotation
        #viz.save(self, voxel.cpu().numpy(), title = "1. Input Data_256x64x64")
        
        x = self.conv1(voxel)
        #viz.save(self, x.cpu().numpy(), title = "2. 3D Conv_128x32x32")
        #print(f"After conv1 shape: {x.shape}")

        x = self.conv2(x)
        #viz.save(self, x.cpu().numpy(), title = "3. 3D Conv_64x16x16")
        #print(f"After conv2 shape: {x.shape}")

        x = self.conv3(x)
        #viz.save(self, x.cpu().numpy(), title = "4. 3D Conv_32x8x8")
        #print(f"After conv3 shape: {x.shape}")

        x = self.conv4(x)
        #viz.save(self, x.cpu().numpy(), title = "5. 3D Conv_16x4x4")
        #print(f"After conv4 shape: {x.shape}")

        x = torch.flatten(x, start_dim=1)
        #print(f"After flatten: {x.shape}")
        
        # onehot_lables concat to data
        #trans_labels = Labeling.angle(self, labels)
        #print(f"trans labels state: {trans_labels.shape} {trans_labels.device}")
        #xcatlabels = torch.cat((x, trans_labels), dim=1)
        #print(f"After labels concat: {xcatlabels.shape}")
        
        # Flatten and apply linear + sigmoid
        #x = self.out(xcatlabels)
        #viz.save(self, x.cpu().numpy(), title = "6. Latent")
        #print(f"After linear shape: {x.shape}")

        fc_dis = self.fc_dis(x)
        fc_aux = self.fc_aux(x)
        classes = self.softmax(fc_aux)
        realfake = self.sigmoid(fc_dis).view(-1, 1).squeeze(1)

        return realfake, classes
        
        
        #return x
    
    def load(self, backup):
        for m_from, m_to in zip(backup.modules(), self.modules()):
             if isinstance(m_to, nn.Linear):
                m_to.weight.data = m_from.weight.data.clone()
                if m_to.bias is not None:
                    m_to.bias.data = m_from.bias.data.clone()

In [ ]:
### check Discriminator shape
## Uncomment print() line in the Discriminator class
'''
discriminator = Discriminator().to(device)
noise = torch.randn(32,1,256,64,64).to(device)

with torch.no_grad():
    test = discriminator(noise, labels)
    print(test.shape)
'''

After conv1 shape: torch.Size([32, 8, 128, 32, 32])
After conv2 shape: torch.Size([32, 32, 64, 16, 16])
After conv3 shape: torch.Size([32, 128, 32, 8, 8])
After conv4 shape: torch.Size([32, 256, 16, 4, 4])
After flatten: torch.Size([32, 65536])
trans labels state: torch.Size([32, 4]) cuda:0
After labels concat: torch.Size([32, 65540])
After linear shape: torch.Size([32, 1])
torch.Size([32, 1])


In [ ]:
### Init model

generator = Generator().to(device)
discriminator = Discriminator().to(device)
viz = Visualization()

# 손실 함수 및 옵티마이저 설정
#criterion = nn.BCELoss()  # Binary Cross Entropy Loss
dis_criterion = nn.BCELoss()
aux_criterion = nn.NLLLoss()
avg_loss_D = 0.0
avg_loss_G = 0.0
avg_loss_A = 0.0

optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.000005, betas=(0.5, 0.999))

# hyperpara
num_epochs = 80  # 학습 횟수
batch_size = 32
latent_dim = 400
real_value = 1
fake_value = 0
d_steps = 1
g_steps = 5
unrolled_steps = 5
samples = []


In [ ]:
### Unrolling

def d_loop(batch, labels):
    optimizer_D.zero_grad()

    #  1A: Train D on real
    dis_output, aux_output = discriminator(batch, labels)
    #d_real_decision = discriminator(batch, labels)
    aux_label = labels # 라벨의 형태가 정수형이며, 연속된 수여야함 변경 필요
    dis_label = torch.full_like(batch_size, real_value, device=device)
    #target = torch.full_like(d_real_decision, real_value, device=device)
    
    dis_errD_real = dis_criterion(dis_output, dis_label)
    aux_errD_real = aux_criterion(aux_output, aux_label)
    #d_real_error = criterion(d_real_decision, target)
    errD_real = dis_errD_real + aux_errD_real
    errD_real.backward()
    D_x = dis_output.detach().mean().item()
    #accuracy = compute_acc(aux_output, aux_label)

    #  1B: Train D on fake
    d_gen_input = torch.randn(batch.size(0), latent_dim).to(device)
    with torch.no_grad():
        d_fake_data = generator(d_gen_input, labels) 
    dix_output, aux_output = discriminator(d_gen_input.detach(), labels.detach())
    #d_fake_decision = discriminator(d_fake_data, labels)
    dis_label = torch.full_like(batch_size, fake_value, device=device)
    aux_label = 
    #target = torch.full_like(d_fake_decision, fake_value, device=device)
    dis_errD_fake = dis_criterion(dis_output, dis_label)
    aux_errD_fake = aux_criterion(aux_output, aux_label)
    d_fake_error = criterion(d_fake_decision, target)  # zeros = fake
    
    d_loss = d_real_error + d_fake_error
    d_loss.backward()
    optimizer_D.step()     # Only optimizes D's parameters; changes based on stored gradients from backward()
    return d_real_error.cpu().item(), d_fake_error.cpu().item()

def d_unrolled_loop(d_gen_input=None):
    optimizer_D.zero_grad()

    #  1A: Train D on real
    d_real_decision = discriminator(batch,labels)
    target = torch.full_like(d_real_decision, real_value, device=device)
    d_real_error = criterion(d_real_decision, target)

    #  1B: Train D on fake
    if d_gen_input is None:
        d_gen_input = torch.randn(batch.size(0), latent_dim).to(device)
    
    with torch.no_grad():
        d_fake_data = generator(d_gen_input,labels)
    d_fake_decision = discriminator(d_fake_data,labels)
    target = torch.full_like(d_fake_decision, fake_value, device=device)
    d_fake_error = criterion(d_fake_decision, target)  # zeros = fake
    
    d_loss = d_real_error + d_fake_error
    d_loss.backward(create_graph=True)
    optimizer_D.step()     # Only optimizes D's parameters; changes based on stored gradients from backward()
    return d_real_error.cpu().item(), d_fake_error.cpu().item()

def g_loop():
    # 2. Train G on D's response (but DO NOT train D on these labels)
    optimizer_G.zero_grad()

    gen_input = torch.randn(batch.size(0), latent_dim).to(device)
        
    if unrolled_steps > 0:
        backup = copy.deepcopy(discriminator)
        for i in range(unrolled_steps):
            d_unrolled_loop(d_gen_input=gen_input)
    
    g_fake_data = generator(gen_input,labels)
    dg_fake_decision = discriminator(g_fake_data,labels)
    target = torch.full_like(dg_fake_decision, real_value, device=device)
    g_error = criterion(dg_fake_decision, target)  # we want to fool, so pretend it's all genuine
    g_error.backward()
    optimizer_G.step()  # Only optimizes G's parameters
    
    if unrolled_steps > 0:
        discriminator.load(backup)    
        del backup
    return g_error.cpu().item()

def g_sample():
    with torch.no_grad():
        gen_input = torch.randn(batch.size(0), latent_dim).to(device)
        gen_input_lables = torch.tensor([0,15,30,45]*8).to(device)
        g_fake_data = generator(gen_input, gen_input_lables)
        return g_fake_data

In [ ]:
### train

for epoch in range(num_epochs):
    PATH = r"D:\research\7. TSP_3\20250216 python용 onepool 추출\augmentation\model_save\generator_epoch{}.pth".format(epoch)

    for batch, labels in dataloader:
        batch, labels = batch.to(device), labels.to(device)
    
        #for it in tqdm_notebook(range(num_iterations)):
        d_infos = []
        for d_index in range(d_steps):
            d_info = d_loop(batch, labels)
            d_infos.append(d_info)
        d_infos = np.mean(d_infos, 0)
        d_real_loss, d_fake_loss = d_infos
    
        g_infos = []
        for g_index in range(g_steps):
            g_info = g_loop()
            g_infos.append(g_info)
        g_infos = np.mean(g_infos)
        g_loss = g_infos

    print(f"Epoch [{epoch + 1} / {num_epochs}],d_real_loss: {d_real_loss}, d_fake_loss: {d_fake_loss}, g_loss: {g_loss}")

    if epoch % 10 == 0:
        g_fake_data = g_sample()
        samples.append(g_fake_data)
        viz.save(g_fake_data[0][0], title=f"{epoch}_condition_0")
        viz.save(g_fake_data[1][0], title=f"{epoch}_condition_15")
        viz.save(g_fake_data[2][0], title=f"{epoch}_condition_30")
        viz.save(g_fake_data[3][0], title=f"{epoch}_condition_45")

        torch.save({
            'epoch': epoch,
            'model_state_dict': generator.state_dict(),
            'optimizer_state_dict': optimizer_G.state_dict(),
            'loss': g_loss
        }, PATH)

In [ ]:
### last epoch save
torch.save({
            'epoch': 80,
            'model_state_dict': generator.state_dict(),
            'optimizer_state_dict': optimizer_G.state_dict(),
            'loss': g_loss
        }, PATH)

g_fake_data = g_sample()
samples.append(g_fake_data)
viz.save(g_fake_data[0][0], title=f"{epoch}_condition_0")
viz.save(g_fake_data[1][0], title=f"{epoch}_condition_15")
viz.save(g_fake_data[2][0], title=f"{epoch}_condition_30")
viz.save(g_fake_data[3][0], title=f"{epoch}_condition_45")

#samples = [8,32,1,256,64,64]
# 8 = epoch number, 32 = batch (0,4,8,...: 0 / 1,5,9,...: 15 / 2, 6, 10,...: 30 / 3, 7, 11,...:45)
samples_torch = torch.tensor(samples)
samples_training = samples_torch.cpu().detach().numpy()
np.save(r"D:\research\7. TSP_3\20250216 python용 onepool 추출\augmentation\g_result\samples_training.npy", samples_training)